# Gene expression prediction with the Genomic Intelligence model package

This notebook deploys the Genomic Intelligence gene expression model from AWS Marketplace in your own account, predicts expression for two sample genes in two cell types on a real-time endpoint, runs the same requests as a batch transform job, and deletes everything it created.

**Before you run it**

1. Subscribe to the product on AWS Marketplace (search for *Genomic Intelligence Gene Expression Prediction*), then choose **Continue to configuration**. The notebook finds the model package for your Region itself.
2. Run it from SageMaker Studio or a notebook instance whose execution role can create SageMaker models, endpoints and transform jobs, and read and write the Region's default SageMaker bucket.
3. You need quota for one `ml.g5.xlarge` endpoint and one `ml.g5.xlarge` transform job in your Region.

**Cost.** The endpoint bills from the moment it is `InService` until the cleanup cell deletes it, at the SageMaker instance rate plus the product's software price. Run the cleanup cell even if something fails.

Research use only. Not for diagnostic or clinical use.

In [ ]:
import json, time
from pathlib import Path

import boto3

session = boto3.session.Session()
region = session.region_name
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")
account = session.client("sts").get_caller_identity()["Account"]

try:
    import sagemaker
    role = sagemaker.get_execution_role()
except Exception:
    raise SystemExit("Run this from SageMaker Studio or a notebook instance, or set `role` to an execution role ARN.")

bucket = f"sagemaker-{region}-{account}"
prefix = "gi-expression-example"
INSTANCE = "ml.g5.xlarge"
print(region, role, bucket)

## The model package for your Region

AWS Marketplace publishes one model package ARN per Region. The table below is filled in when the listing is published.

In [ ]:
# Version 1.1.0, from the listing's launch page: {region: model package ARN}.
MODEL_PACKAGE_ARNS = {
    "ap-northeast-1": "arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "ap-northeast-2": "arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "ap-south-1": "arn:aws:sagemaker:ap-south-1:077584701553:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "ap-southeast-1": "arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "ap-southeast-2": "arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "ca-central-1": "arn:aws:sagemaker:ca-central-1:470592106596:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "eu-central-1": "arn:aws:sagemaker:eu-central-1:446921602837:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "eu-north-1": "arn:aws:sagemaker:eu-north-1:136758871317:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "eu-west-1": "arn:aws:sagemaker:eu-west-1:985815980388:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "eu-west-2": "arn:aws:sagemaker:eu-west-2:856760150666:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "eu-west-3": "arn:aws:sagemaker:eu-west-3:843114510376:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "sa-east-1": "arn:aws:sagemaker:sa-east-1:270155090741:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "us-east-1": "arn:aws:sagemaker:us-east-1:865070037744:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "us-east-2": "arn:aws:sagemaker:us-east-2:057799348421:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "us-west-1": "arn:aws:sagemaker:us-west-1:382657785993:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
    "us-west-2": "arn:aws:sagemaker:us-west-2:594846645681:model-package/gi-expression-8192-v2-71b4fddd40cc300fae3ed43e653cccf5",
}

if region not in MODEL_PACKAGE_ARNS:
    raise SystemExit(
        f"No model package ARN for {region} yet. Supported: {sorted(MODEL_PACKAGE_ARNS) or 'none listed'}. "
        "Copy the ARN shown under 'Continue to configuration' on the listing into MODEL_PACKAGE_ARNS."
    )
model_package_arn = MODEL_PACKAGE_ARNS[region]
print(model_package_arn)

## Sample requests

Two genes, each in two cell lines: APOA1 (a liver gene) and HBB (beta-globin), in HepG2 (liver) and K562 (erythroleukemia). Each request is a GRCh38 region from 40,960 bp upstream of the transcription start site through the end of the gene, in the gene's orientation, with:

- `tss_index`: position of the transcription start site in `sequence`
- `tes_index` (optional, recommended): end of the gene in `sequence`, exclusive
- `options.description`: the cell type, in the exact format the model was trained on (see the published cell-type list)

In [ ]:
DATA = Path("data")
samples = {p.stem: json.loads(p.read_text()) for p in sorted((DATA / "input").glob("*.json"))}
for name, body in samples.items():
    print(f"{name}: {len(body['sequence'])} bp, tss_index {body['tss_index']}, tes_index {body.get('tes_index', 'not given')}")

## Deploy a real-time endpoint

The model runs with network isolation, as every Marketplace model package does. This takes several minutes.

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"gi-expression-{stamp}"
endpoint_name = f"gi-expression-{stamp}"

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=endpoint_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InstanceType": INSTANCE,
        "InitialInstanceCount": 1,
    }],
)
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
print("InService:", endpoint_name)

## Predict

Expected shape: APOA1 high in HepG2 and low in K562; HBB the other way round.

In [ ]:
results = {}
for name, body in samples.items():
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    results[name] = json.loads(response["Body"].read())
    p = results[name]["prediction"]
    print(f"{name}: {p['expression_log_tpm']:.2f} ln(TPM+1), {p['expression_tpm']:.1f} TPM")

Compare with the published responses. Values can differ slightly from them depending on the GPU.

In [ ]:
for name, r in results.items():
    expected = json.loads((DATA / "output" / f"{name}.json").read_text())
    delta = abs(r["prediction"]["expression_log_tpm"] - expected["prediction"]["expression_log_tpm"])
    print(f"{name}: difference {delta:.3f} ln(TPM+1)")

## Batch transform

Each S3 object is one request body, passed whole to the model (`SplitType: None`), and each response comes back as its own `.out` object.

In [ ]:
for name, body in samples.items():
    s3.put_object(Bucket=bucket, Key=f"{prefix}/input/{name}.json", Body=json.dumps(body).encode())

job_name = f"gi-expression-batch-{stamp}"
sm.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    MaxConcurrentTransforms=1,
    BatchStrategy="SingleRecord",
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "AssembleWith": "None"},
    TransformResources={"InstanceType": INSTANCE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=job_name)
print(job_name, sm.describe_transform_job(TransformJobName=job_name)["TransformJobStatus"])

for name in samples:
    obj = s3.get_object(Bucket=bucket, Key=f"{prefix}/output/{name}.json.out")
    r = json.loads(obj["Body"].read())
    print(name, round(r["prediction"]["expression_log_tpm"], 2))

## Clean up

Deletes the endpoint (which stops its billing), the endpoint configuration and the model.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
sm.delete_model(ModelName=model_name)
print("deleted", endpoint_name)

To stop the subscription, open **Your Marketplace software** in the AWS Marketplace console, choose the product, and cancel the subscription.